# Lab 14: Claims and reconciliation

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

In [ ]:
import inspect
print(inspect.getsource(schemas.MgmtClaim))
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
LARGE = llm.P["llm"]["models"]["large"]
extracted = {b: llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks[b]["text"]), LARGE) for b in sd.CLAIM_BLOCKS}
claims = sd.claim_rows(extracted)          # claim ids; CY2025 aligned to FY2025 with basis "calendar"
Path("outputs/work").mkdir(parents=True, exist_ok=True)
with open("outputs/work/claims.jsonl", "w") as fh:
    for c in claims: fh.write(json.dumps(c) + "\n")
import pandas as pd
pd.DataFrame(claims)[["claim_id", "metric", "value_abs", "period_label", "period_basis", "qualifier"]]

In [ ]:
# From the slide: Reconciliation: claims against seller financials
def reconcile(claim, facts, tol=0.01, high=0.05):
    """Tier-4 claim vs tier-3 facts -> output record."""
    rec = {"metric": claim.metric, "period": claim.period,
           "claim_id": claim.claim_id, "fact_id": None,
           "status": None, "gap": None, "severity": None}
    f = facts.get((claim.metric, claim.period))
    if f is None:
        rec["status"] = "no_fact"; return rec
    rec["fact_id"] = f["fact_id"]
    if claim.qualifier in ("run-rate", "pro forma"):
        rec["status"] = "definition_mismatch"; return rec
    if claim.period_basis != f["period_basis"]:
        rec["status"] = "period_mismatch"; return rec
    v = claim.value * claim.scale
    gap = (v - f["value"]) / abs(f["value"])
    rec["gap"] = round(gap, 6)
    if abs(gap) <= tol:
        rec["status"] = "match"; return rec
    rec["status"] = "discrepancy"
    rec["severity"] = "high" if abs(gap) > high else "medium"
    return rec

In [ ]:
from types import SimpleNamespace as NS
facts = {(f["metric"], f["period"]): f for f in sd.seller_facts()}
R = llm.P["reconcile"]
recs = [reconcile(NS(**c), facts, tol=R["tolerance_rel"], high=R["high_gap_rel"]) for c in claims]
Path("outputs/reconcile").mkdir(parents=True, exist_ok=True)
with open("outputs/reconcile/deal_a.jsonl", "w") as fh:     # the CS2 section-8 format
    for r in recs: fh.write(json.dumps(r) + "\n")
pd.DataFrame(recs)

In [ ]:
import yaml
key = yaml.safe_load(open("data/dataroom/deal_a/answer_key.yaml"))
expected = {t["claim_block"]: t["expected_status"] for t in key["traps"]}
by_block = {c["block_id"]: r for c, r in zip(claims, recs) if c["block_id"] in expected and r["status"] != "match"}
hits = sum(by_block.get(b, {}).get("status") == s for b, s in expected.items())
flagged = sum(r["status"] != "match" for r in recs)
print(f"trap recall {hits}/{len(expected)} | precision {hits}/{flagged}")

**Exercise 14.** Why does the run-rate claim come back `definition_mismatch` rather than `discrepancy`? Change the reconcile order so the gap is computed first: what goes wrong?